# HTFRS Fase A: Time-Aware Collaborative Filtering

Implementación de las Ecuaciones 1-3 del paper Rostami et al. (2023).

- **Eq. 1**: User Similarity ponderada por Time Weight
- **Eq. 2**: Time Weight (función exponencial)
- **Eq. 3**: CF-based Prediction

**Output**: Predicciones CF para cada par (user, item) → se usa en el notebook de ensamblaje.

In [1]:
import numpy as np
import pandas as pd
import os
import sys
import pickle
from collections import defaultdict
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.model_selection import train_test_split
import time as time_module

# Parámetros del paper
LAMBDA = 2.5       # Time factor parameter (paper optimal)
K_NEIGHBORS = 50    # Top-K vecinos por usuario
K_RECS = 10         # Top-K recomendaciones
MIN_CORATINGS = 3   # Mínimo de co-ratings para calcular similitud
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Muestreo: None = todos los usuarios, o un int para muestrear
MAX_USERS = None  # Cambiar a ej. 10000 si es muy lento

print('Setup listo')

Setup listo


## 1. Carga de datos

In [2]:
# --- Carga de Reviews ---
try:
    import kagglehub
    path = kagglehub.dataset_download('irkaal/foodcom-recipes-and-reviews')
    reviews = pd.read_csv(os.path.join(path, 'reviews.csv'))
    print(f'Reviews cargados desde Kaggle: {len(reviews)}')
except Exception:
    reviews = pd.read_csv('../Dataset_recetas/reviews.csv')
    print(f'Reviews cargados localmente: {len(reviews)}')

# --- Carga de Recipes ---
try:
    import gdown
    url = 'https://drive.google.com/uc?id=1Rl8XowC9N6cxrdiPvH4wToFUvZrrBqRG'
    gdown.download(url, 'recipes_final_consolidado.csv', quiet=True)
    recipes = pd.read_csv('recipes_final_consolidado.csv')
except Exception:
    recipes = pd.read_csv('../Dataset_recetas/recipes_final_consolidado.csv')
print(f'Recipes cargados: {len(recipes)}')

100%|██████████| 723M/723M [00:05<00:00, 134MB/s]

Extracting files...


Reviews cargados desde Kaggle: 1401982
Recipes cargados: 522359


In [3]:
# Parsear ExtractedServingSize
recipes['ExtractedServingSize'] = recipes['ExtractedServingSize'].str.extract(r'\((.*?)\)').astype(float)

# Filtrar recetas con serving size <= 0
recipes = recipes[recipes['ExtractedServingSize'] > 0].copy()

# Filtrar usuarios con solo 1 review
user_counts = reviews['AuthorId'].value_counts()
valid_users = user_counts[user_counts > 1].index
reviews_filtrado = reviews[reviews['AuthorId'].isin(valid_users)].copy()

# Solo reviews de recetas válidas
valid_recipes = set(recipes['RecipeId'])
reviews_filtrado = reviews_filtrado[reviews_filtrado['RecipeId'].isin(valid_recipes)].copy()

print(f'Usuarios: {reviews_filtrado["AuthorId"].nunique()}')
print(f'Recetas: {reviews_filtrado["RecipeId"].nunique()}')
print(f'Ratings: {len(reviews_filtrado)}')

Usuarios: 72098
Recetas: 257707
Ratings: 1199589


In [4]:
# Calcular sellos
recipes['IsHighCalories'] = ((recipes['Calories'] / recipes['ExtractedServingSize']) * 100) >= 275
recipes['IsHighSugar'] = ((recipes['SugarContent'] / recipes['ExtractedServingSize']) * 100) >= 10
recipes['IsHighSaturatedFat'] = ((recipes['SaturatedFatContent'] / recipes['ExtractedServingSize']) * 100) >= 4
recipes['IsHighSodium'] = ((recipes['SodiumContent'] / recipes['ExtractedServingSize']) * 100) >= 400

sello_cols = ['IsHighCalories', 'IsHighSugar', 'IsHighSaturatedFat', 'IsHighSodium']
recipes['num_sellos'] = recipes[sello_cols].sum(axis=1).astype(int)
recipe_sellos_dict = dict(zip(recipes['RecipeId'], recipes['num_sellos']))

print(f'Distribución de sellos:')
print(recipes['num_sellos'].value_counts().sort_index())

Distribución de sellos:
num_sellos
0    233920
1    128100
2     74977
3     74173
4      9801
Name: count, dtype: int64


## 2. Train/Test Split (80/20 random)

In [5]:
train_df, test_df = train_test_split(
    reviews_filtrado, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

# Mapeos de IDs a índices
all_users = sorted(reviews_filtrado['AuthorId'].unique())
all_items = sorted(reviews_filtrado['RecipeId'].unique())

user2idx = {u: i for i, u in enumerate(all_users)}
idx2user = {i: u for u, i in user2idx.items()}
item2idx = {it: i for i, it in enumerate(all_items)}
idx2item = {i: it for it, i in item2idx.items()}

n_users = len(all_users)
n_items = len(all_items)
print(f'n_users={n_users}, n_items={n_items}')

# Muestreo opcional de usuarios
if MAX_USERS and MAX_USERS < n_users:
    user_activity = train_df['AuthorId'].value_counts()
    sampled_users = set(user_activity.head(MAX_USERS).index)
    print(f'Muestreando {MAX_USERS} usuarios más activos')
else:
    sampled_users = set(all_users)
    print(f'Usando todos los {n_users} usuarios')

Train: 959671, Test: 239918
n_users=72098, n_items=257707
Usando todos los 72098 usuarios


## 3. Discretización temporal

In [6]:
train_df = train_df.copy()
train_df['DateSubmitted'] = pd.to_datetime(train_df['DateSubmitted'], errors='coerce')
train_df = train_df.dropna(subset=['DateSubmitted'])

date_min = train_df['DateSubmitted'].min()
date_max = train_df['DateSubmitted'].max()

# t en meses desde la fecha mínima
train_df['t_months'] = (
    (train_df['DateSubmitted'] - date_min).dt.days / 30.0
)
TP = (date_max - date_min).days / 30.0

print(f'Rango temporal: {date_min.date()} a {date_max.date()}')
print(f'TP = {TP:.1f} meses ({TP/12:.1f} años)')

Rango temporal: 2000-02-25 a 2020-12-27
TP = 253.7 meses (21.1 años)


## 4. Estructuras de datos para eficiencia

In [7]:
# Estructura por usuario: {user_id: {item_id: (rating, t_months)}}
user_ratings = defaultdict(dict)
for row in train_df[['AuthorId', 'RecipeId', 'Rating', 't_months']].itertuples(index=False):
    user_ratings[row[0]][row[1]] = (row[2], row[3])

# Promedio de ratings por usuario
user_mean_rating = {}
for u, items in user_ratings.items():
    ratings = [r for r, t in items.values()]
    user_mean_rating[u] = np.mean(ratings)

# Índice invertido: item -> set de usuarios que lo ratearon
item_to_users = defaultdict(set)
for u, items in user_ratings.items():
    if u not in sampled_users:
        continue
    for item_id in items:
        item_to_users[item_id].add(u)

print(f'Usuarios con ratings: {len(user_ratings)}')
print(f'Items en índice invertido: {len(item_to_users)}')

Usuarios con ratings: 70916
Items en índice invertido: 234335


## 5. Time-Aware User Similarity (Eqs. 1-2)

**Eq. 2**: $TW(u,v,i) = e^{-\lambda \cdot (TP - t_{u,i})/TP} \times e^{-\lambda \cdot (TP - t_{v,i})/TP}$

**Eq. 1**: Pearson correlation ponderada por TW

In [8]:
def time_weight(t_ui, t_vi, TP, lam=LAMBDA):
    """Eq. 2: ratings recientes pesan más."""
    return np.exp(-lam * (TP - t_ui) / TP) * np.exp(-lam * (TP - t_vi) / TP)


def user_similarity_tw(u, v, user_ratings, user_mean_rating, TP, lam=LAMBDA):
    """Eq. 1: Pearson correlation ponderada por Time Weight."""
    items_u = user_ratings[u]
    items_v = user_ratings[v]
    common = set(items_u.keys()) & set(items_v.keys())

    if len(common) < MIN_CORATINGS:
        return 0.0

    mean_u = user_mean_rating[u]
    mean_v = user_mean_rating[v]

    num = 0.0
    den_u = 0.0
    den_v = 0.0

    for item in common:
        r_ui, t_ui = items_u[item]
        r_vi, t_vi = items_v[item]
        tw = time_weight(t_ui, t_vi, TP, lam)

        diff_u = r_ui - mean_u
        diff_v = r_vi - mean_v

        num += diff_u * diff_v * tw
        den_u += diff_u ** 2 * tw
        den_v += diff_v ** 2 * tw

    if den_u == 0 or den_v == 0:
        return 0.0

    return num / (np.sqrt(den_u) * np.sqrt(den_v))

In [9]:
def find_neighbors(target_user, user_ratings, item_to_users, user_mean_rating, TP,
                   k=K_NEIGHBORS, lam=LAMBDA):
    """Encuentra los top-K vecinos más similares usando índice invertido."""
    # Candidatos: usuarios que comparten al menos 1 item
    candidates = set()
    for item_id in user_ratings[target_user]:
        if item_id in item_to_users:
            candidates.update(item_to_users[item_id])
    candidates.discard(target_user)

    # Calcular similitud con cada candidato
    sims = []
    for candidate in candidates:
        sim = user_similarity_tw(target_user, candidate, user_ratings, user_mean_rating, TP, lam)
        if sim != 0:
            sims.append((candidate, sim))

    # Top-K por similitud absoluta
    sims.sort(key=lambda x: abs(x[1]), reverse=True)
    return dict(sims[:k])


# Calcular vecinos para todos los usuarios muestreados
print(f'Calculando vecinos para {len(sampled_users)} usuarios...')
start_time = time_module.time()

user_neighbors = {}  # {user_id: {neighbor_id: similarity}}
users_list = sorted(sampled_users)

for i, u in enumerate(users_list):
    if u not in user_ratings:
        continue
    user_neighbors[u] = find_neighbors(
        u, user_ratings, item_to_users, user_mean_rating, TP
    )
    if (i + 1) % 1000 == 0:
        elapsed = time_module.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(users_list) - i - 1) / rate
        print(f'  {i+1}/{len(users_list)} usuarios ({rate:.1f} users/s, ~{remaining/60:.0f} min restantes)')

elapsed = time_module.time() - start_time
print(f'Vecinos calculados en {elapsed/60:.1f} minutos')
print(f'Usuarios con al menos 1 vecino: {sum(1 for v in user_neighbors.values() if v)}')

Calculando vecinos para 72098 usuarios...
  1000/72098 usuarios (26.2 users/s, ~45 min restantes)
  2000/72098 usuarios (28.8 users/s, ~41 min restantes)
  3000/72098 usuarios (25.9 users/s, ~44 min restantes)
  4000/72098 usuarios (26.8 users/s, ~42 min restantes)
  5000/72098 usuarios (26.8 users/s, ~42 min restantes)
  6000/72098 usuarios (28.6 users/s, ~39 min restantes)
  7000/72098 usuarios (28.7 users/s, ~38 min restantes)
  8000/72098 usuarios (29.7 users/s, ~36 min restantes)
  9000/72098 usuarios (29.3 users/s, ~36 min restantes)
  10000/72098 usuarios (28.6 users/s, ~36 min restantes)
  11000/72098 usuarios (29.1 users/s, ~35 min restantes)
  12000/72098 usuarios (29.2 users/s, ~34 min restantes)
  13000/72098 usuarios (29.3 users/s, ~34 min restantes)
  14000/72098 usuarios (29.6 users/s, ~33 min restantes)
  15000/72098 usuarios (30.0 users/s, ~32 min restantes)
  16000/72098 usuarios (30.0 users/s, ~31 min restantes)
  17000/72098 usuarios (30.4 users/s, ~30 min restantes

## 6. CF-based Prediction (Eq. 3)

$p^{cf}_{iu} = \bar{r}_u + \frac{\sum_{v \in C_u} SimU(u,v) \times (r_{iv} - \bar{r}_v)}{\sum_{v \in C_u} |SimU(u,v)|}$

In [10]:
def predict_cf(user_id, item_id, user_neighbors, user_ratings, user_mean_rating, global_mean):
    """Eq. 3: Predicción CF basada en vecinos."""
    if user_id not in user_neighbors or not user_neighbors[user_id]:
        return global_mean

    mean_u = user_mean_rating.get(user_id, global_mean)
    neighbors = user_neighbors[user_id]

    num = 0.0
    den = 0.0

    for v, sim in neighbors.items():
        if v not in user_ratings or item_id not in user_ratings[v]:
            continue
        r_iv, _ = user_ratings[v][item_id]
        mean_v = user_mean_rating.get(v, global_mean)
        num += sim * (r_iv - mean_v)
        den += abs(sim)

    if den == 0:
        return mean_u

    pred = mean_u + num / den
    return np.clip(pred, 0, 5)

## 7. Generar Top-K recomendaciones

In [11]:
global_mean = train_df['Rating'].mean()

# Items de entrenamiento por usuario (para excluir de recomendaciones)
train_items_per_user = defaultdict(set)
for row in train_df[['AuthorId', 'RecipeId']].itertuples(index=False):
    train_items_per_user[row[0]].add(row[1])

# Usuarios de test que están en nuestro sample
test_users = set(test_df['AuthorId'].unique()) & sampled_users & set(user_neighbors.keys())
print(f'Usuarios de test a evaluar: {len(test_users)}')

# Pool de candidatos: items más populares en train
item_popularity = train_df['RecipeId'].value_counts()
candidate_items = list(item_popularity.head(5000).index)
print(f'Pool de candidatos: {len(candidate_items)} items')

Usuarios de test a evaluar: 41698
Pool de candidatos: 5000 items


In [12]:
print('Generando recomendaciones...')
start_time = time_module.time()

user_recommendations = {}  # {user_id: [top-K item_ids]}
test_users_list = sorted(test_users)

for i, u in enumerate(test_users_list):
    seen = train_items_per_user.get(u, set())
    candidates = [it for it in candidate_items if it not in seen]

    preds = {}
    for it in candidates:
        preds[it] = predict_cf(u, it, user_neighbors, user_ratings, user_mean_rating, global_mean)

    # Top-K por predicted rating
    sorted_items = sorted(preds.items(), key=lambda x: x[1], reverse=True)
    top_k = [item_id for item_id, _ in sorted_items[:K_RECS]]

    user_recommendations[u] = top_k

    if (i + 1) % 500 == 0:
        elapsed = time_module.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(test_users_list) - i - 1) / rate
        print(f'  {i+1}/{len(test_users_list)} usuarios ({rate:.1f} users/s, ~{remaining/60:.0f} min restantes)')

elapsed = time_module.time() - start_time
print(f'Recomendaciones generadas en {elapsed/60:.1f} minutos')

Generando recomendaciones...
  500/41698 usuarios (40.4 users/s, ~17 min restantes)
  1000/41698 usuarios (40.6 users/s, ~17 min restantes)
  1500/41698 usuarios (40.1 users/s, ~17 min restantes)
  2000/41698 usuarios (39.5 users/s, ~17 min restantes)
  2500/41698 usuarios (40.0 users/s, ~16 min restantes)
  3000/41698 usuarios (40.2 users/s, ~16 min restantes)
  3500/41698 usuarios (41.0 users/s, ~16 min restantes)
  4000/41698 usuarios (41.7 users/s, ~15 min restantes)
  4500/41698 usuarios (41.7 users/s, ~15 min restantes)
  5000/41698 usuarios (41.8 users/s, ~15 min restantes)
  5500/41698 usuarios (41.8 users/s, ~14 min restantes)
  6000/41698 usuarios (42.0 users/s, ~14 min restantes)
  6500/41698 usuarios (42.1 users/s, ~14 min restantes)
  7000/41698 usuarios (42.0 users/s, ~14 min restantes)
  7500/41698 usuarios (42.6 users/s, ~13 min restantes)
  8000/41698 usuarios (42.7 users/s, ~13 min restantes)
  8500/41698 usuarios (43.4 users/s, ~13 min restantes)
  9000/41698 usuario

## 8. Evaluación

In [13]:
# === evaluation_utils (inline para Colab) ===
import numpy as np

RELEVANCE_THRESHOLD = 4

def get_relevant_items(test_df, user_id, threshold=RELEVANCE_THRESHOLD):
    user_test = test_df[test_df['AuthorId'] == user_id]
    return set(user_test[user_test['Rating'] >= threshold]['RecipeId'])

def precision_at_k(recommended, relevant, k=10):
    return len(set(recommended[:k]) & set(relevant)) / k

def recall_at_k(recommended, relevant, k=10):
    rel_set = set(relevant)
    if len(rel_set) == 0:
        return 0.0
    return len(set(recommended[:k]) & rel_set) / len(rel_set)

def f1_at_k(recommended, relevant, k=10):
    p = precision_at_k(recommended, relevant, k)
    r = recall_at_k(recommended, relevant, k)
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)

def _dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if relevances.size:
        return np.sum((2**relevances - 1) / np.log2(np.arange(2, relevances.size + 2)))
    return 0.0

def ndcg_at_k(recommended, relevant, k=10):
    relevances = [1 if r in relevant else 0 for r in recommended[:k]]
    dcg = _dcg_at_k(relevances, k)
    ideal = _dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0

def average_precision_at_k(recommended, relevant, k=10):
    hits = 0
    sum_prec = 0.0
    rel_set = set(relevant)
    for i, item in enumerate(recommended[:k]):
        if item in rel_set:
            hits += 1
            sum_prec += hits / (i + 1)
    return sum_prec / min(len(rel_set), k) if rel_set else 0.0

def sellos_at_k(recommended, recipe_sellos_dict, k=10):
    sellos = [recipe_sellos_dict.get(r, 0) for r in recommended[:k]]
    return np.mean(sellos) if sellos else 0.0

def sello_free_at_k(recommended, recipe_sellos_dict, k=10):
    sellos = [recipe_sellos_dict.get(r, 0) for r in recommended[:k]]
    return np.mean([1 if s == 0 else 0 for s in sellos]) if sellos else 0.0

def evaluate_recommendations(recommended, relevant, recipe_sellos_dict, k=10):
    return {
        'P@K': precision_at_k(recommended, relevant, k),
        'R@K': recall_at_k(recommended, relevant, k),
        'F1@K': f1_at_k(recommended, relevant, k),
        'nDCG@K': ndcg_at_k(recommended, relevant, k),
        'MAP@K': average_precision_at_k(recommended, relevant, k),
        'S@K': sellos_at_k(recommended, recipe_sellos_dict, k),
        'SS@K': sello_free_at_k(recommended, recipe_sellos_dict, k),
    }

def evaluate_all_users(user_recommendations, test_df, recipe_sellos_dict,
                       k=10, threshold=RELEVANCE_THRESHOLD):
    all_metrics = []
    for user_id, recommended in user_recommendations.items():
        relevant = get_relevant_items(test_df, user_id, threshold)
        metrics = evaluate_recommendations(recommended, relevant, recipe_sellos_dict, k)
        all_metrics.append(metrics)
    if not all_metrics:
        return {}
    return {key: np.mean([m[key] for m in all_metrics]) for key in all_metrics[0]}

# --- Evaluar ---
metrics = evaluate_all_users(
    user_recommendations, test_df, recipe_sellos_dict, k=K_RECS
)

print('\n=== Resultados Time-Aware CF (Eqs. 1-3) ===')
print(f'Usuarios evaluados: {len(user_recommendations)}')
print(f'K = {K_RECS}')
print(f'λ = {LAMBDA}, K_neighbors = {K_NEIGHBORS}')
print()
for metric, value in metrics.items():
    print(f'  {metric}: {value:.6f}')


=== Resultados Time-Aware CF (Eqs. 1-3) ===
Usuarios evaluados: 41698
K = 10
λ = 2.5, K_neighbors = 50

  P@K: 0.003441
  R@K: 0.014147
  F1@K: 0.004341
  nDCG@K: 0.017176
  MAP@K: 0.006059
  S@K: 1.137513
  SS@K: 0.392076


## 9. Guardar resultados para el notebook de ensamblaje

In [14]:
output = {
    'user_recommendations_cf': user_recommendations,
    'user_neighbors': {u: dict(list(v.items())[:10]) for u, v in user_neighbors.items()},
    'user_mean_rating': user_mean_rating,
    'metrics_cf': metrics,
    'params': {
        'lambda': LAMBDA,
        'k_neighbors': K_NEIGHBORS,
        'k_recs': K_RECS,
        'test_size': TEST_SIZE,
        'random_state': RANDOM_STATE,
        'max_users': MAX_USERS,
    },
}

import pickle
with open('htfrs_cf_output.pkl', 'wb') as f:
    pickle.dump(output, f)

print('Resultados guardados en htfrs_cf_output.pkl')
print(f'Tamaño: {os.path.getsize("htfrs_cf_output.pkl") / 1e6:.1f} MB')

Resultados guardados en htfrs_cf_output.pkl
Tamaño: 8.0 MB
